# Lexorion — DeBERTa fine-tuning on a free Colab GPU

Fine-tunes **one multi-label DeBERTa-v3-base** (8 sigmoid heads, one per risk category)
on CUAD paragraphs, with the project's recall-first thresholding policy.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

**Total runtime:** ~50-70 min (data rebuild ~10 min, training ~35-50 min).
The last cell downloads the two result files you carry back to the repo —
the 700MB model checkpoint stays here unless you want it.

In [ ]:
# 1. Confirm the GPU is on (should print a Tesla T4 table)
!nvidia-smi

In [ ]:
# 2. Get the code
!git clone --depth 1 https://github.com/DhavalVibhakar99/Lexorion-Cuad.git
%cd Lexorion-Cuad

In [ ]:
# 3. Dependencies (Colab already ships torch + CUDA).
#    transformers pinned <5: v5 casts model weights to fp16 under fp16=True,
#    which crashes AMP gradient unscaling on T4.
%pip install -q "transformers<5" accelerate datasets sentencepiece protobuf pyyaml pyarrow

In [ ]:
# 4. Rebuild the CUAD paragraph dataset from the public source (~10 min)
!python -m src.data_pipeline.download_cuad
!python -m src.data_pipeline.parse_cuad
!python -m src.data_pipeline.chunk_contracts

In [ ]:
# 5. Train (single multi-label run: BCE + per-class pos_weight, fp16, 256 tokens)
#    Prints the per-category recall-first results table at the end.
!python -m src.models.clause_detector --epochs 2 --batch_size 16

In [ ]:
# 6. Download the two result files to carry back to the repo
from google.colab import files
files.download('data/processed/deberta_predictions.parquet')
files.download('data/processed/deberta_training_summary.csv')

## Back on your machine

Drop both files into `data/processed/`, then:

```bash
./venv/bin/python -m src.evaluation.model_comparison
cp data/processed/model_comparison*.csv examples/
```

The DEBERTA row appears in the comparison automatically (full test split,
true prevalence — directly comparable to the BASELINE and EMBED rows).

**Optional — keep the model:** zip and download `checkpoints/deberta_multilabel/`
(~700MB) or push it to your Hugging Face account:

```python
from huggingface_hub import login, HfApi
login()  # paste a write token
HfApi().upload_folder(folder_path='checkpoints/deberta_multilabel/best_model',
                      repo_id='<your-hf-user>/lexorion-deberta', repo_type='model')
```